# P2.2 — Sequence-LoRA Candidate v1.3.1 — ONE CLICK S42 (RunPod A40)

Status: **READY**. This notebook trains/resumes `BGL → HDFS → HADOOP` with seed
`42`. Every checkpoint and completed target is persisted below
`/workspace/SeqLogAD/sequence-v13r1/P2.2`. It never loads the P2.1 Semantic
adapter or resumes a v1.0/v1.1 Sequence recovery snapshot.

Before clicking **Run All**:

1. Deploy one NVIDIA **A40 48 GB** Pod with a Network Volume at `/workspace`.
   Use `runpod/pytorch:1.0.3-cu1281-torch291-ubuntu2404`.
2. Map a RunPod secret to environment variable `HF_TOKEN`.
3. Upload `seqlogad-code-all-experts-s42-v13r1.zip`, `phase2-sequence-s42.zip`, `phase2-source-development-v13r1.zip` and `projmem-0.4.0a1-py3-none-any.whl` to
   `/workspace/uploads`.
4. Upload/open this notebook in JupyterLab and click **Run All** once.

After interruption, restart with the same Network Volume and click **Run All**
again. Stop only after `SAFE_TO_STOP_RUNTIME=YES`.

V13R1 records source-development metrics in Projmem. Keep the entire persistent directory, including `.pmem`, `attempts` and exports. Before Projmem installs, failures are retained in the journal and registered on the next successful setup. Live progress is in `attempts/<id>/training.log`.


In [ ]:
# HUMAN CONFIG — no fold/seed editing is required.
from pathlib import Path
import os, uuid

TASK_ID = "P2.2"
NOTEBOOK_STATUS = "READY"
BLOCKERS = []
RUN_MODE = "development"
SEED = 42
TARGETS = ["BGL", "HDFS", "HADOOP"]
UPLOAD_ROOT = Path("/workspace/uploads")
STATE_ROOT = Path("/workspace/SeqLogAD")
PERSISTENT_ROOT = STATE_ROOT / "sequence-v13r1"
HF_HOME = STATE_ROOT / "hf-cache"
VENV = STATE_ROOT / "venv-py312"
RUNTIME_ROOT = STATE_ROOT / "runtime" / "ad0f313b1b2e-4ed76b92e4b0-c765eb88a4fe"
REPO_ROOT = str(RUNTIME_ROOT / "seqlogad_code_all_experts_s42")
DATA_ROOT = str(RUNTIME_ROOT / "phase2_sequence")
SOURCE_DEVELOPMENT_ROOT = str(RUNTIME_ROOT / "phase2_source_development_v13r1")
EXPECTED_SOURCE_DEVELOPMENT_ZIP_SHA256 = 'c765eb88a4feebc8ceb157110e5a6bd28dec942a7fb9343d7f6b969b3e79b295'
EXPECTED_SOURCE_DEVELOPMENT_MANIFEST_SHA256 = '885b53821671e7f60e6df5270a78a23535900bf4ba6d339e69b29f93eb2c6399'
SESSION_ROOT = Path("/tmp/seqlogad_sessions") / uuid.uuid4().hex
OUTPUT_ROOT = str(SESSION_ROOT / "seqlogad_outputs")
EXPECTED_CODE_ZIP_SHA256 = 'ad0f313b1b2e482f75eb2f5d10800fe59c451a2d1adc0c9e85aa0f1b0d01efb3'
EXPECTED_DATA_ZIP_SHA256 = '4ed76b92e4b001138b28b4b3b4d3017d6ff7f30dd31e8cca0649bfdf34ede781'
EXPECTED_PMEM_WHEEL_SHA256 = '637964eed3af38c6170c5571bbc02338268cf31bddb15eaf3cc6d883c6b3ce4f'
EXPECTED_BUNDLE_SHA256 = "1268b669a8b1b474658cfa7daa6556c987b0f31f07e0101ed30af250a8d7f129"
REQUIRED_GPU = "NVIDIA A40"
MIN_VRAM_GIB = 40

assert RUN_MODE == "development" and SEED == 42
assert TARGETS == ["BGL", "HDFS", "HADOOP"]
assert NOTEBOOK_STATUS == "READY", "Notebook is fail-closed: " + ", ".join(BLOCKERS)
assert UPLOAD_ROOT.is_dir(), "RunPod Network Volume must be mounted at /workspace"
for directory in (STATE_ROOT, PERSISTENT_ROOT, HF_HOME, SESSION_ROOT):
    directory.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_HOME)
print({"session_root": str(SESSION_ROOT), "persistent_root": str(PERSISTENT_ROOT)})

# Journal exists before runtime setup, including failures before Projmem installs.
import json, datetime, traceback
ATTEMPT_ROOT = PERSISTENT_ROOT / "attempts" / SESSION_ROOT.name
ATTEMPT_ROOT.mkdir(parents=True, exist_ok=False)
JOURNAL_PATH = ATTEMPT_ROOT / "lifecycle.jsonl"

def journal(event, detail=""):
    token = os.environ.get("HF_TOKEN")
    if token:
        detail = detail.replace(token, "[TOKEN_HIDDEN]")
    with JOURNAL_PATH.open("a") as handle:
        handle.write(json.dumps({"event": event, "detail": detail,
            "utc": datetime.datetime.now(datetime.timezone.utc).isoformat()}) + "\n")
        handle.flush()
        os.fsync(handle.fileno())

journal("STARTED")


## Apply the verified RunPod uv compatibility repair


In [ ]:
journal("UV_STARTED")
try:
    # RunPod images may install `uv` outside sys.executable's directory.
    import shutil, sys
    from pathlib import Path
    
    uv_real = shutil.which("uv")
    if uv_real:
        uv_expected = Path(sys.executable).parent / "uv"
        if not uv_expected.exists():
            uv_expected.symlink_to(uv_real)
        print("UV_FIX=PASS", {"uv_real": uv_real, "uv_expected": str(uv_expected)})
    else:
        print("UV_FIX=DEFERRED_UNTIL_BOOTSTRAP")
except BaseException:
    journal("UV_FAILED", traceback.format_exc())
    if "pmem_cli" in globals() and Path(pmem_cli).is_file():
        run_live([pmem_cli, "note", "Attempt " + SESSION_ROOT.name + " failed in UV; see durable journal.",
                  "--tag", "attempt-failure"], check=False, cwd=str(PERSISTENT_ROOT))
    raise
journal("UV_COMPLETED")


## Verify uploads, extract atomically and build the frozen runtime


In [ ]:
journal("SETUP_STARTED")
try:
    from pathlib import PurePosixPath
    import hashlib, json, shutil, stat, subprocess, sys, zipfile
    
    def digest(path):
        value = hashlib.sha256()
        with Path(path).open("rb") as handle:
            for block in iter(lambda: handle.read(1024 * 1024), b""):
                value.update(block)
        return value.hexdigest()
    
    def locate(expected, pattern="*.zip"):
        matches = [path for path in UPLOAD_ROOT.glob(pattern) if digest(path) == expected]
        assert len(matches) == 1, f"Expected one upload with SHA-256 {expected}; found {matches}"
        return matches[0]
    
    def safe_extract(archive, destination):
        destination = Path(destination).resolve()
        with zipfile.ZipFile(archive) as handle:
            infos = handle.infolist()
            names = [item.filename for item in infos]
            assert len(names) == len(set(names)), "Duplicate ZIP member"
            for item in infos:
                member = PurePosixPath(item.filename)
                assert not member.is_absolute() and ".." not in member.parts and "\\" not in item.filename
                assert ((item.external_attr >> 16) & 0o170000) != stat.S_IFLNK, "ZIP symlink forbidden"
            assert handle.testzip() is None, "ZIP CRC failure"
            handle.extractall(destination)
    
    code_zip = locate(EXPECTED_CODE_ZIP_SHA256)
    data_zip = locate(EXPECTED_DATA_ZIP_SHA256)
    pmem_wheel = locate(EXPECTED_PMEM_WHEEL_SHA256, "*.whl")
    label_zip = locate(EXPECTED_SOURCE_DEVELOPMENT_ZIP_SHA256)
    marker = RUNTIME_ROOT / "EXTRACTION.json"
    expected_marker = {"code_sha256": EXPECTED_CODE_ZIP_SHA256, "data_sha256": EXPECTED_DATA_ZIP_SHA256, "source_dev_sha256": EXPECTED_SOURCE_DEVELOPMENT_ZIP_SHA256}
    if RUNTIME_ROOT.exists():
        assert marker.is_file(), f"Unverified runtime directory exists: {RUNTIME_ROOT}"
        assert json.loads(marker.read_text()) == expected_marker, "Runtime extraction identity mismatch"
    else:
        partial = RUNTIME_ROOT.with_name(RUNTIME_ROOT.name + ".partial")
        if partial.exists():
            shutil.rmtree(partial)
        partial.mkdir(parents=True)
        safe_extract(code_zip, partial)
        safe_extract(data_zip, partial)
        safe_extract(label_zip, partial)
        assert (partial / "seqlogad_code_all_experts_s42").is_dir()
        assert (partial / "phase2_sequence/manifests/bundle.json").is_file()
        assert (partial / "phase2_source_development_v13r1/manifest.json").is_file()
        (partial / "EXTRACTION.json").write_text(json.dumps(expected_marker, sort_keys=True) + "\n")
        partial.rename(RUNTIME_ROOT)
    
    assert Path(REPO_ROOT).is_dir()
    assert Path(DATA_ROOT, "manifests/bundle.json").is_file()
    assert digest(Path(SOURCE_DEVELOPMENT_ROOT) / "manifest.json") == EXPECTED_SOURCE_DEVELOPMENT_MANIFEST_SHA256
    print({"code_zip": str(code_zip), "data_zip": str(data_zip), "extraction": "PASS"})
    
    def isolated_env():
        env = dict(os.environ)
        env.pop("PYTHONPATH", None)
        env["PYTHONNOUSERSITE"] = "1"
        env["PYTHONUNBUFFERED"] = "1"
        return env
    
    def run_live(args, check=True, cwd=None, capture=False):
        output = []
        with subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1, env=isolated_env(), cwd=cwd) as process:
            for line in process.stdout:
                if capture:
                    output.append(line)
                token = os.environ.get("HF_TOKEN")
                if token:
                    line = line.replace(token, "[TOKEN_HIDDEN]")
                print(line, end="", flush=True)
            code = process.wait()
        if check and code:
            raise subprocess.CalledProcessError(code, args)
        return (code, "".join(output)) if capture else code
    
    run_live(["nvidia-smi"])
    PYTHON = str(VENV / "bin/python")
    if not Path(PYTHON).is_file():
        if sys.version_info[:2] == (3, 12):
            run_live([sys.executable, "-m", "pip", "install", "virtualenv==20.31.2"])
            run_live([sys.executable, "-m", "virtualenv", "--python", sys.executable, str(VENV)])
        else:
            run_live([sys.executable, "-m", "pip", "install", "uv==0.8.17"])
            uv_real = shutil.which("uv")
            assert uv_real, "uv bootstrap executable missing"
            uv_expected = Path(sys.executable).parent / "uv"
            if not uv_expected.exists():
                uv_expected.symlink_to(uv_real)
            run_live([str(uv_expected), "python", "install", "3.12"])
            run_live([str(uv_expected), "venv", "--python", "3.12", "--seed", str(VENV)])
    
    run_live([PYTHON, "-c", "import sys; assert sys.version_info[:2] == (3, 12); print(sys.version)"])
    
    run_live([PYTHON, "-m", "pip", "install", str(pmem_wheel)])
    pmem_cli = str(VENV / "bin/pmem")
    run_live([pmem_cli, "init", "--name", "SeqLogAD Sequence V13R1",
              "--metric", "source_development_macro_f1", "--metric-direction", "max"],
             cwd=str(PERSISTENT_ROOT))
    for prior in sorted((PERSISTENT_ROOT / "attempts").glob("*/lifecycle.jsonl")):
        stamp = prior.parent / "journal-registered"
        if stamp.exists() or prior == JOURNAL_PATH:
            continue
        run_live([pmem_cli, "track", str(prior.relative_to(PERSISTENT_ROOT)), "--update"],
                 cwd=str(PERSISTENT_ROOT))
        run_live([pmem_cli, "note", "Recovered prior attempt journal: " + str(prior.parent.name)
                  + "; inspect final event and training-state.json. No completion is inferred.",
                  "--tag", "recovered-attempt"], cwd=str(PERSISTENT_ROOT))
        stamp.write_text("registered\n")
    run_live([pmem_cli, "note", "Attempt started: " + SESSION_ROOT.name,
              "--tag", "attempt-start"], cwd=str(PERSISTENT_ROOT))
    journal("PROJMEM_INITIALIZED")
    
    run_live([PYTHON, "-m", "pip", "install", "torch==2.9.1", "--index-url",
              "https://download.pytorch.org/whl/cu128"])
    run_live([PYTHON, "-m", "pip", "install", "-e", REPO_ROOT, "packaging>=24,<27"])
    run_live([PYTHON, "-c", "from seqlogad.semantic.runtime import install_missing; install_missing(" + repr(REPO_ROOT) + ")"])
    
    hardware_check = f"""
    import json, torch
    assert torch.cuda.is_available(), "CUDA is unavailable"
    name = torch.cuda.get_device_name(0)
    free, total = torch.cuda.mem_get_info(0)
    native_bf16 = torch.cuda.is_bf16_supported(including_emulation=False)
    assert {REQUIRED_GPU!r} in name, f"Expected {REQUIRED_GPU}, got {{name}}"
    assert total >= {MIN_VRAM_GIB} * 1024**3, f"Expected at least {MIN_VRAM_GIB} GiB VRAM"
    assert native_bf16, "Native BF16 is required"
    print(json.dumps({{"gpu": name, "free_vram_bytes": free, "total_vram_bytes": total,
                      "native_bf16": native_bf16, "torch": torch.__version__,
                      "cuda": torch.version.cuda}}, indent=2))
    """
    run_live([PYTHON, "-c", hardware_check])
    run_live([PYTHON, "-m", "seqlogad.sequence.cli", "environment", "--repo-root", REPO_ROOT])
except BaseException:
    journal("SETUP_FAILED", traceback.format_exc())
    if "pmem_cli" in globals() and Path(pmem_cli).is_file():
        run_live([pmem_cli, "note", "Attempt " + SESSION_ROOT.name + " failed in SETUP; see durable journal.",
                  "--tag", "attempt-failure"], check=False, cwd=str(PERSISTENT_ROOT))
    raise
journal("SETUP_COMPLETED")


## Secure Hugging Face authentication


In [ ]:
journal("AUTH_STARTED")
try:
    token = os.environ.get("HF_TOKEN")
    assert token, "Set HF_TOKEN as a RunPod environment secret; never paste it into the notebook"
    run_live([PYTHON, "-m", "seqlogad.sequence.cli", "auth", "--repo-root", REPO_ROOT])
except BaseException:
    journal("AUTH_FAILED", traceback.format_exc())
    if "pmem_cli" in globals() and Path(pmem_cli).is_file():
        run_live([pmem_cli, "note", "Attempt " + SESSION_ROOT.name + " failed in AUTH; see durable journal.",
                  "--tag", "attempt-failure"], check=False, cwd=str(PERSISTENT_ROOT))
    raise
journal("AUTH_COMPLETED")


## Sequential target loop with persistent recovery


In [ ]:
journal("TRAIN_STARTED")
try:
    import re
    
    args = [
        PYTHON, "-m", "seqlogad.phase2_execution.cli",
        "--task-id", TASK_ID,
        "--repo-root", REPO_ROOT,
        "--data-root", DATA_ROOT,
        "--output-root", OUTPUT_ROOT,
        "--persistent-root", str(PERSISTENT_ROOT),
        "--expected-bundle-sha256", EXPECTED_BUNDLE_SHA256,
        "--sequence-candidate", "v13r1",
        "--source-development-root", SOURCE_DEVELOPMENT_ROOT,
        "--expected-source-development-sha256", EXPECTED_SOURCE_DEVELOPMENT_MANIFEST_SHA256,
    ]
    pmem_cli_exit, pmem_result = run_live(
        [pmem_cli, "run", "--name", "P2.2 Sequence V13R1 RunPod attempt",
         "--seed", "42", "--metrics", str((ATTEMPT_ROOT / "metrics.json").relative_to(PERSISTENT_ROOT)),
         "--", PYTHON, "-m", "seqlogad.phase2_execution.durable_attempt",
         "--attempt", str(ATTEMPT_ROOT), "--persistent-root", str(PERSISTENT_ROOT), "--", *args],
        check=False, cwd=str(PERSISTENT_ROOT), capture=True,
    )
    pmem_success = pmem_cli_exit == 0 and bool(
        re.search(r"(?m)^Run run_[0-9a-f]+ success$", pmem_result)
    )
    bundle_path = PERSISTENT_ROOT / "projmem_exports" / f"{SESSION_ROOT.name}.json"
    bundle_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        if pmem_success:
            run_live([pmem_cli, "track", "P2.2/seqlogad_outputs_P2.2_S42_FINAL.zip",
                      "--update"], cwd=str(PERSISTENT_ROOT))
    finally:
        run_live([pmem_cli, "export-bundle", "--out", str(bundle_path)],
                 cwd=str(PERSISTENT_ROOT))
    print({"projmem_db": str(PERSISTENT_ROOT / ".pmem/pmem.db"),
           "projmem_export": str(bundle_path)})
    assert pmem_success, "Projmem recorded a failed Sequence V13R1 execution; inspect its run and export"
except BaseException:
    journal("TRAIN_FAILED", traceback.format_exc())
    if "pmem_cli" in globals() and Path(pmem_cli).is_file():
        run_live([pmem_cli, "note", "Attempt " + SESSION_ROOT.name + " failed in TRAIN; see durable journal.",
                  "--tag", "attempt-failure"], check=False, cwd=str(PERSISTENT_ROOT))
    raise
journal("TRAIN_COMPLETED")


## Final persistent-integrity and safe-stop gate


In [ ]:
journal("EXPORT_STARTED")
try:
    final_zip = PERSISTENT_ROOT / TASK_ID / "seqlogad_outputs_P2.2_S42_FINAL.zip"
    final_sidecar = Path(str(final_zip) + ".sha256")
    assert final_zip.is_file() and final_sidecar.is_file(), "Verified final P2.2 ZIP is missing"
    declared_hash, declared_name = final_sidecar.read_text().strip().split("  ", 1)
    assert declared_name == final_zip.name and digest(final_zip) == declared_hash
    shutil.rmtree(SESSION_ROOT)
    print("====================================")
    print("SEQLOGAD P2.2 RUNPOD EXECUTION SUMMARY")
    print(f"FINAL_ZIP={final_zip}")
    print(f"FINAL_ZIP_SIZE={final_zip.stat().st_size}")
    print(f"FINAL_ZIP_SHA256={declared_hash}")
    print("PERSISTENT_STORAGE_VERIFIED=YES")
    print("SAFE_TO_STOP_RUNTIME=YES")
    print("====================================")
    
    journal("COMPLETED")
    run_live([pmem_cli, "track", str(JOURNAL_PATH.relative_to(PERSISTENT_ROOT)), "--update"], cwd=str(PERSISTENT_ROOT))
except BaseException:
    journal("EXPORT_FAILED", traceback.format_exc())
    if "pmem_cli" in globals() and Path(pmem_cli).is_file():
        run_live([pmem_cli, "note", "Attempt " + SESSION_ROOT.name + " failed in EXPORT; see durable journal.",
                  "--tag", "attempt-failure"], check=False, cwd=str(PERSISTENT_ROOT))
    raise
journal("EXPORT_COMPLETED")
